In [ ]:
# Imports and configuration
import os
import json
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, List

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuration
DATA_DIR = "data/dataset/tcga_kirc_nii/BD"  # Change this to your data directory
DECISIONS_FILE = DATA_DIR + "/volume_decisions.json"  # Output file for decisions
PERCENTILE_LOW = 1   # Lower percentile for intensity normalization
PERCENTILE_HIGH = 99 # Upper percentile for intensity normalization

print(f"Configuration loaded. Data directory: {DATA_DIR}")

In [ ]:
# File discovery and data structures

def find_nifti_files(data_dir: str) -> List[Path]:
    """Recursively find all .nii.gz files, sorted deterministically."""
    data_path = Path(data_dir)
    if not data_path.exists():
        print(f"Warning: {data_dir} does not exist!")
        return []
    
    files = sorted(data_path.rglob("*.nii.gz"))
    print(f"Found {len(files)} .nii.gz files")
    return files


def load_volume(filepath: Path) -> Optional[np.ndarray]:
    """Load a NIfTI volume and return as numpy array.
    
    Assumes the last axis is Z (slices). Handles common orientations.
    Returns None if loading fails.
    """
    try:
        img = nib.load(str(filepath))
        data = img.get_fdata()
        
        # Ensure 3D volume
        if data.ndim == 4:
            # Take first timepoint/channel if 4D
            data = data[..., 0]
        
        # Assumption: last axis is Z (slices)
        # Shape: (X, Y, Z) or (Z, Y, X) - we'll treat last as Z
        return data
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None


def normalize_slice(slice_data: np.ndarray, plow: int = 1, phigh: int = 99) -> np.ndarray:
    """Normalize slice intensity using percentile clipping."""
    if slice_data.size == 0:
        return slice_data
    hu_scaled = np.clip(slice_data, -200, 300)
    vmin, vmax = np.percentile(hu_scaled, [plow, phigh])
    if vmax > vmin:
        normalized = np.clip(hu_scaled, vmin, vmax)
        normalized = (normalized - vmin) / (vmax - vmin)
    else:
        normalized = np.zeros_like(hu_scaled)
    
    return normalized


def load_decisions(filepath: str) -> Dict[str, str]:
    """Load decisions from JSON file if it exists."""
    if not os.path.exists(filepath):
        return {}
    
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
            return {item['filepath']: item['decision'] for item in data}
    except Exception as e:
        print(f"Error loading decisions: {e}")
        return {}


def save_decisions(decisions: Dict[str, str], filepath: str):
    """Save decisions to JSON file with timestamp."""
    output = [
        {
            'filepath': path,
            'decision': decision,
            'timestamp': datetime.now().isoformat()
        }
        for path, decision in decisions.items()
    ]
    
    try:
        with open(filepath, 'w') as f:
            json.dump(output, f, indent=2)
        print(f"Saved {len(decisions)} decisions to {filepath}")
    except Exception as e:
        print(f"Error saving decisions: {e}")


print("Utility functions loaded.")

In [ ]:
# Initialize application state

class VolumeReviewState:
    """Application state and data cache."""
    
    def __init__(self, data_dir: str, decisions_file: str):
        self.files = find_nifti_files(data_dir)
        self.num_cases = len(self.files)
        
        # Current indices
        self.case_idx = 0
        self.slice_idx = 0
        
        # Cached volume data
        self.current_volume: Optional[np.ndarray] = None
        self.current_file: Optional[Path] = None
        
        # Decisions: filepath -> "keep" or "delete"
        self.decisions: Dict[str, str] = load_decisions(decisions_file)
        
        # Undo history: list of (filepath, old_decision or None)
        self.history: List[tuple] = []
        
        self.decisions_file = decisions_file
    
    def get_current_filepath(self) -> Optional[str]:
        """Get current file path as string."""
        if 0 <= self.case_idx < self.num_cases:
            return str(self.files[self.case_idx])
        return None
    
    def get_current_decision(self) -> str:
        """Get decision for current file."""
        filepath = self.get_current_filepath()
        if filepath:
            return self.decisions.get(filepath, "UNDECIDED")
        return "UNDECIDED"
    
    def load_current_volume(self) -> bool:
        """Load the current volume into cache. Returns True on success."""
        if not (0 <= self.case_idx < self.num_cases):
            return False
        
        filepath = self.files[self.case_idx]
        
        # Check if already loaded
        if self.current_file == filepath and self.current_volume is not None:
            return True
        
        # Load new volume
        self.current_volume = load_volume(filepath)
        self.current_file = filepath
        
        if self.current_volume is not None:
            # Reset slice to middle
            num_slices = self.current_volume.shape[-1]
            self.slice_idx = num_slices // 2
            return True
        
        return False
    
    def get_num_slices(self) -> int:
        """Get number of slices in current volume (Z axis for axial)."""
        if self.current_volume is not None:
            # Return Z dimension for axial view
            return self.current_volume.shape[2] if len(self.current_volume.shape) >= 3 else 1
        return 1
    
    def get_current_slice(self) -> Optional[tuple]:
        """Get current 2D slices (axial variable, coronal fixed at middle)."""
        if self.current_volume is None:
            return None
        
        # Get dimensions
        shape = self.current_volume.shape
        y_dim = shape[1] if len(shape) >= 2 else 1
        z_dim = shape[2] if len(shape) >= 3 else 1
        
        # Axial: variable slice along Z axis (controlled by slider)
        slice_idx_z = max(0, min(self.slice_idx, z_dim - 1))
        axial_slice = self.current_volume[..., slice_idx_z]
        
        # Coronal: FIXED at middle Y slice
        slice_idx_y_fixed = y_dim // 2
        coronal_slice = self.current_volume[:, slice_idx_y_fixed, :]
        
        return (axial_slice, coronal_slice, slice_idx_z, z_dim)
    
    def set_decision(self, decision: str, advance: bool = True):
        """Set decision for current case and optionally advance."""
        filepath = self.get_current_filepath()
        if not filepath:
            return
        
        # Record for undo
        old_decision = self.decisions.get(filepath)
        self.history.append((filepath, old_decision))
        
        # Set new decision
        self.decisions[filepath] = decision
        
        # Auto-save after each decision
        save_decisions(self.decisions, self.decisions_file)
        
        # Advance to next case
        if advance and self.case_idx < self.num_cases - 1:
            self.case_idx += 1
            self.load_current_volume()
    
    def undo_last_decision(self):
        """Undo the most recent decision change."""
        if not self.history:
            print("No decisions to undo")
            return
        
        filepath, old_decision = self.history.pop()
        
        if old_decision is None:
            # Was undecided, remove entry
            self.decisions.pop(filepath, None)
        else:
            # Restore old decision
            self.decisions[filepath] = old_decision
        
        print(f"Undone: {filepath}")





# Create state instanceprint(f"Initialized with {state.num_cases} cases")
state = VolumeReviewState(DATA_DIR, DECISIONS_FILE)
print(f"Loaded {len(state.decisions)} existing decisions")

In [ ]:
# UI Widgets and display function

output = widgets.Output()

slice_slider = widgets.IntSlider(
    value=0, min=0, max=1, step=1,
    description='Slice:',
    continuous_update=False,
    layout=widgets.Layout(width='600px')
)

btn_prev_case = widgets.Button(description='◀ Previous Case', button_style='info')
btn_next_case = widgets.Button(description='Next Case ▶', button_style='info')
btn_keep = widgets.Button(description='✓ Keep', button_style='success')
btn_delete = widgets.Button(description='✗ Delete', button_style='danger')
btn_undo = widgets.Button(description='↶ Undo', button_style='warning')


def render_display():
    """Render the current volume slice and status information."""
    output.clear_output(wait=True)
    with output:
        if state.num_cases == 0:
            print("No .nii.gz files found in the data directory.")
            return
        
        if not state.load_current_volume() or state.current_volume is None:
            print(f"Failed to load: {state.get_current_filepath()}")
            return
        
        slice_data = state.get_current_slice()
        if slice_data is None:
            print("No slice data available")
            return
        
        axial_slice, coronal_slice, slice_idx_z, z_dim = slice_data
        
        # Normalize for display
        norm_axial = normalize_slice(axial_slice, PERCENTILE_LOW, PERCENTILE_HIGH)
        norm_coronal = normalize_slice(coronal_slice, PERCENTILE_LOW, PERCENTILE_HIGH)
        
        # Create figure
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
        
        # Axial view (left) - VARIABLE
        ax1.imshow(norm_axial.T, cmap='gray', origin='lower', aspect='auto')
        ax1.set_title(f"Axial View - Slice {slice_idx_z + 1}/{z_dim}", fontsize=14)
        ax1.axis('off')
        
        # Coronal view (right) - FIXED with horizontal line
        ax2.imshow(norm_coronal.T, cmap='gray', origin='lower', aspect='auto')
        ax2.set_title("Coronal View (Fixed - Middle Y)", fontsize=14)
        ax2.axis('off')
        
        # Yellow line indicating axial position
        h, w = norm_coronal.T.shape
        line_y = (slice_idx_z / max(1, z_dim - 1)) * h
        ax2.axhline(y=line_y, color='yellow', linewidth=3, alpha=0.8)
        
        plt.tight_layout()
        plt.show()
        plt.close(fig)
        
        # Status info
        filepath = state.get_current_filepath()
        print(f"\n📁 File: {Path(filepath).name}")
        print(f"📂 Path: {filepath}")
        print(f"📊 Shape: {state.current_volume.shape}")
        print(f"📋 Decision: {state.get_current_decision().upper()}")
        print(f"\n📍 Case {state.case_idx + 1}/{state.num_cases}")
        print(f"📈 Decisions made: {len(state.decisions)}")


def update_slider():
    """Update slider range and value WITHOUT triggering callback."""
    slice_slider.unobserve(on_slice_change, names='value')
    slice_slider.max = max(0, state.get_num_slices() - 1)
    slice_slider.value = state.slice_idx
    slice_slider.observe(on_slice_change, names='value')


# Callbacks
def on_slice_change(change):
    state.slice_idx = change['new']
    render_display()

def on_prev_case(_):
    if state.case_idx > 0:
        state.case_idx -= 1
        state.load_current_volume()
        update_slider()
        render_display()

def on_next_case(_):
    if state.case_idx < state.num_cases - 1:
        state.case_idx += 1
        state.load_current_volume()
        update_slider()
        render_display()

def on_keep(_):
    state.set_decision("keep", advance=True)
    update_slider()
    render_display()

def on_delete(_):
    state.set_decision("delete", advance=True)
    update_slider()
    render_display()

def on_undo(_):
    state.undo_last_decision()
    render_display()

# Attach callbacks
slice_slider.observe(on_slice_change, names='value')
btn_prev_case.on_click(on_prev_case)
btn_next_case.on_click(on_next_case)
btn_keep.on_click(on_keep)
btn_delete.on_click(on_delete)
btn_undo.on_click(on_undo)

print("UI ready.")

In [ ]:
# Display the UI (run cells 4 and 5 first if you restart kernel)

ui = widgets.VBox([
    widgets.HBox([btn_prev_case, btn_next_case, btn_keep, btn_delete, btn_undo],
                 layout=widgets.Layout(justify_content='center', margin='10px')),
    widgets.HBox([slice_slider], layout=widgets.Layout(justify_content='center', margin='10px')),
    output
])

# Initialize and render
if state.num_cases > 0:
    state.load_current_volume()
    update_slider()
    render_display()

display(ui)

In [ ]:
import shutil

def move_deleted_files_to_recycle_bin():
    data_path = Path(DATA_DIR)
    parts = data_path.parts
    
    dataset_idx = parts.index('dataset')
    dataset_base = parts[dataset_idx + 1].replace('_nii', '').replace('_seg', '')
    class_name = parts[-1]
    base_path = Path(*parts[:dataset_idx+1])
    
    deleted_files = [fp for fp, dec in state.decisions.items() if dec == "delete"]
    if not deleted_files:
        return
    
    nii_recycle = base_path / f"{dataset_base}_nii" / 'recycle_bin' / class_name
    seg_recycle = base_path / f"{dataset_base}_seg" / 'recycle_bin' / class_name
    nii_recycle.mkdir(parents=True, exist_ok=True)
    seg_recycle.mkdir(parents=True, exist_ok=True)
    
    moved = 0
    for filepath in deleted_files:
        nii_filename = Path(filepath).name
        # Remove _0000 suffix to get seg filename: case_id_0000.nii.gz -> case_id.nii.gz
        seg_filename = nii_filename.replace('_0000.nii.gz', '.nii.gz')
        
        # Move from _nii folder
        for f in (base_path / f"{dataset_base}_nii" / class_name).glob(nii_filename):
            if 'recycle_bin' not in f.parts:
                shutil.move(str(f), str(nii_recycle / f.name))
                moved += 1
        
        # Move from _seg folder (different filename)
        for f in (base_path / f"{dataset_base}_seg" / class_name).glob(seg_filename):
            if 'recycle_bin' not in f.parts:
                shutil.move(str(f), str(seg_recycle / f.name))
                moved += 1
    
    print(f"Moved {moved} files")

# move_deleted_files_to_recycle_bin()